# Day 04 — Hash Maps, Visualized

A hash function turns a key into a bucket index. Collisions are inevitable; with linear
probing we just walk to the next open slot. Let's watch it, then see why we must resize.

## Keys land in buckets; collisions probe forward

In [ ]:
CAP = 8
buckets = [None] * CAP

def insert(k):
    i = k % CAP; probes = 0
    while buckets[i] is not None:
        i = (i + 1) % CAP; probes += 1
    buckets[i] = k
    return i, probes

for k in [5, 13, 21, 3, 11]:   # 5, 13, 21 all hash to slot 5 -> collisions
    slot, p = insert(k)
    print(f'insert {k:2d} -> slot {slot}   ({p} extra probes)')
print('buckets:', buckets)

## Average probe length explodes as the table fills

This is *why* a hash map resizes once it's ~70% full — beyond that, lookups slow down fast.

In [ ]:
import matplotlib.pyplot as plt, random

def avg_probes(load, cap=1000):
    table = [None] * cap; n = int(load * cap); total = 0
    for _ in range(n):
        i = random.randrange(10**9) % cap; steps = 1
        while table[i] is not None:
            i = (i + 1) % cap; steps += 1
        table[i] = 1; total += steps
    return total / max(n, 1)

loads = [i / 20 for i in range(1, 20)]
plt.figure(figsize=(7, 4))
plt.plot(loads, [avg_probes(l) for l in loads], marker='o')
plt.axvline(0.7, color='crimson', ls='--', label='resize threshold ~0.7')
plt.xlabel('load factor'); plt.ylabel('avg probes per insert')
plt.title('Average probe length vs. how full the table is')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Takeaways

- Hash → index gives O(1) *average* lookup; collisions are handled by probing.
- Deletion must leave a **tombstone**, or you break other keys' probe chains.
- Keep the load factor under ~0.7 by resizing, or probe chains blow up.

**Now build the real thing** in `homework.py` (with tombstones + resize) and run `pytest -q`.